# Практика встречи 1. Стоит ли тестировать и на чём измерять

**60 минут:** три упражнения по 20 минут, разборы встроены в каждое.
Всё считается офлайн: нужны только `numpy`, `scipy`, `pandas` (и `matplotlib`, если хочется картинку). Сеть не нужна, данные синтетические.

Как устроено каждое упражнение:

1. **Задача** — что нужно получить.
2. **Входные данные** — конкретные числа (не меняются между участниками).
3. **Решение** — полный код, ячейки идут сверху вниз.
4. **Критерий приёмки** — `assert`, которые должны пройти.

Три гипотезы (H1–H3) и параметры магазина совпадают со слайдами блока 1.

---

## Сквозной кейс: интернет-магазин

| Параметр | Значение | Имя в коде |
|---|---|---|
| Конверсия в заказ | 5.0% | `P0` |
| Пользователей на чекауте | 50 000 в неделю | `USERS_WEEK` |
| Выручка | 3 000 000 ₽ в неделю | `REV_WEEK` |
| Горизонт | 12 месяцев = 52 недели | `HORIZON_WEEKS` |

Используемые формулы:

$$\text{break-even} = \frac{\text{тест}+\text{внедрение}+\text{поддержка}}{Rev \times H}, \qquad MDE_{\text{отн}} = \frac{(z_{1-\alpha/2}+z_{1-\beta})\sqrt{2p(1-p)/n}}{p}$$

$$P(\text{внедрить}\mid\theta) = 1-\Phi\!\left(z_{1-\alpha}-\frac{\delta_{true}-\delta_{BE}}{SE}\right)$$

**Допущения модели** (те же, что в курсе I.06): эффект постоянен во времени; выгоды получаем и во время теста (половина трафика), и после внедрения; откат платный, но потери после ошибочного внедрения всё равно идут по горизонту; модель нейтральна к риску.

---

## Упражнение 1. Окупаемость трёх гипотез (20 мин)

### Задача

Для трёх гипотез посчитать и свести в таблицу:

1. точку безубыточности — в % и в ₽ за горизонт;
2. MDE в деньгах при `n = 50 000` на группу и `α = 0.05` (двусторонний, мощность 80%);
3. вероятность внедрения при правдоподобном эффекте `μ`;
4. ожидаемые выгоды, затраты, `RR`, чистую прибыль;
5. вердикт: **тестировать / внедрять без теста / не внедрять, доработать**.

### Входные данные

| | H1. Купон отдельной строкой | H2. Редизайн корзины | H3. Новый платёжный провайдер |
|---|---|---|---|
| Тест, ₽ | 180 000 | 120 000 | 400 000 |
| Внедрение, ₽ | 120 000 | 80 000 | 700 000 |
| Поддержка, ₽/мес | 20 000 | 10 000 | 30 000 |
| Откат, ₽ | 250 000 | 100 000 | 2 500 000 |
| Ожидания | $\mu = 0\%$, $\sigma = 3\%$ | $\mu = +3.3\%$, $\sigma = 1.5\%$ | $\mu = -1\%$, $\sigma = 2\%$ |

Все три гипотезы тестируются на одном и том же магазине.

### Что сдать

Таблицу из пяти строк-метрик и три вердикта. Вердикт ставится так: тест сравниваем не только с «ничем», но и с **внедрением без теста** — `REV_WEEK × μ × HORIZON_WEEKS`.

### Решение

**Ячейка 1. Окружение и параметры кейса**

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from scipy.stats import norm

P0 = 0.05                 # конверсия в заказ
USERS_WEEK = 50_000       # пользователей на чекауте в неделю
REV_WEEK = 3_000_000      # выручка, ₽ в неделю
HORIZON_MONTHS = 12
HORIZON_WEEKS = HORIZON_MONTHS * 52 / 12


**Ячейка 2. Движок: точка безубыточности, вероятность внедрения, деньги**

In [ ]:
@dataclass
class Hypothesis:
    name: str
    test_cost: float
    implementation: float
    rollback: float
    monthly: float
    mu: float
    sigma: float

    @property
    def maintenance(self) -> float:
        return self.monthly * HORIZON_MONTHS

    @property
    def break_even(self) -> float:
        """Относительный uplift, при котором выручка за горизонт покрывает фиксированные затраты."""
        fixed = self.test_cost + self.implementation + self.maintenance
        return fixed / (REV_WEEK * HORIZON_WEEKS)


def ship_probability(h, theta, n, alpha=0.05):
    """P(внедрить) при истинном эффекте theta: порог — точка безубыточности, alpha односторонний."""
    p_test = np.clip(P0 * (1 + theta), 1e-9, 1 - 1e-9)
    delta_true = p_test - P0
    delta_be = P0 * h.break_even
    se = np.sqrt(P0 * (1 - P0) / n + p_test * (1 - p_test) / n)
    return norm.sf(norm.ppf(1 - alpha) - (delta_true - delta_be) / se)


def evaluate(h, n, alpha=0.05):
    """Выгоды и затраты с поправкой на риск: числа модели I.06."""
    theta = np.linspace(-0.12, 0.12, 1200)
    density = norm.pdf(theta, h.mu, h.sigma)
    density = density / np.trapezoid(density, theta)

    p_ship = ship_probability(h, theta, n, alpha)
    test_weeks = 2 * n / USERS_WEEK
    post_weeks = np.maximum(HORIZON_WEEKS - test_weeks, 0)

    test_value = REV_WEEK * theta * test_weeks * 0.5          # половина трафика видит вариант во время теста
    post_value = REV_WEEK * theta * post_weeks * p_ship       # после внедрения — только если внедрили
    decision_cost = (h.implementation + h.maintenance
                     + h.rollback * (theta < h.break_even)) * p_ship

    gains = (np.maximum(test_value, 0) + np.maximum(post_value, 0)) * density
    losses = (np.maximum(-test_value, 0) + np.maximum(-post_value, 0) + decision_cost) * density

    gain = np.trapezoid(gains, theta)
    loss = h.test_cost + np.trapezoid(losses, theta)
    return dict(gain=gain, loss=loss, rr=gain / loss, net=gain - loss, weeks=test_weeks,
                p_ship=float(ship_probability(h, h.mu, n, alpha)),
                gain_test=float(np.trapezoid(np.maximum(test_value, 0) * density, theta)),
                gain_post=float(np.trapezoid(np.maximum(post_value, 0) * density, theta)))


def mde_relative(n, alpha=0.05, power=0.8):
    z = norm.ppf(1 - alpha / 2) + norm.ppf(power)
    return z * np.sqrt(2 * P0 * (1 - P0) / n) / P0


def verdict(h, n, alpha=0.05):
    r = evaluate(h, n, alpha)
    blind = REV_WEEK * h.mu * HORIZON_WEEKS
    if r["net"] <= 0 and blind <= 0:
        return "не внедрять, доработать"
    if blind > r["net"]:
        return "внедрять без теста"
    return "тестировать"


**Ячейка 3. Три гипотезы и таблица**

In [ ]:
H1 = Hypothesis("H1. Купон отдельной строкой", 180_000, 120_000, 250_000, 20_000, 0.000, 0.030)
H2 = Hypothesis("H2. Редизайн корзины", 120_000, 80_000, 100_000, 10_000, 0.033, 0.015)
H3 = Hypothesis("H3. Новый платёжный провайдер", 400_000, 700_000, 2_500_000, 30_000, -0.010, 0.020)

N_TEST, ALPHA = 50_000, 0.05

rows = []
for h in (H1, H2, H3):
    r = evaluate(h, N_TEST, ALPHA)
    mde = mde_relative(N_TEST, ALPHA)
    rows.append({
        "break-even, %": round(100 * h.break_even, 3),
        "break-even, млн ₽": round(h.break_even * REV_WEEK * HORIZON_WEEKS / 1e6, 2),
        "MDE, %": round(100 * mde, 2),
        "MDE, млн ₽": round(mde * REV_WEEK * HORIZON_WEEKS / 1e6, 1),
        "P(внедрить)": round(r["p_ship"], 3),
        "RR": round(r["rr"], 2),
        "тест: net, млн ₽": round(r["net"] / 1e6, 2),
        "без теста, млн ₽": round(REV_WEEK * h.mu * HORIZON_WEEKS / 1e6, 2),
        "вердикт": verdict(h, N_TEST, ALPHA),
    })

table = pd.DataFrame(rows, index=[h.name for h in (H1, H2, H3)])
table


**Ячейка 4. Где именно тест зарабатывает (разбор H1)**

In [ ]:
r1 = evaluate(H1, N_TEST, ALPHA)
components = pd.Series({
    "выгода во время теста, млн ₽": r1["gain_test"] / 1e6,
    "выгода после внедрения (взвешена P внедрения), млн ₽": r1["gain_post"] / 1e6,
    "все затраты, млн ₽": r1["loss"] / 1e6,
    "чистая прибыль, млн ₽": r1["net"] / 1e6,
})
components.round(2)


**Ячейка 5. Критерий приёмки**

In [ ]:
r1, r2, r3 = (evaluate(h, N_TEST, ALPHA) for h in (H1, H2, H3))

# точка безубыточности
assert abs(100 * H1.break_even - 0.346) < 0.002
assert abs(H1.break_even * REV_WEEK * HORIZON_WEEKS - 540_000) < 1_000
assert abs(100 * H3.break_even - 0.936) < 0.002

# MDE в деньгах
assert abs(100 * mde_relative(N_TEST) - 7.72) < 0.05

# H1: тест оправдан, хотя мощность слабая
assert 3.9e5 < r1["net"] < 4.4e5
assert 2.4 < r1["rr"] < 2.8
assert 0.03 < r1["p_ship"] < 0.05
assert verdict(H1, N_TEST) == "тестировать"

# H2: тест только откладывает внедрение
assert 1.6e6 < r2["net"] < 2.2e6
assert abs(REV_WEEK * H2.mu * HORIZON_WEEKS - 5_148_000) < 1_000
assert verdict(H2, N_TEST) == "внедрять без теста"

# H3: и тест, и внедрение убыточны
assert r3["net"] < 0 and r3["rr"] < 0.3
assert verdict(H3, N_TEST) == "не внедрять, доработать"

print("Упражнение 1: все проверки пройдены")

### Ответы

| | H1. Купон | H2. Редизайн | H3. Провайдер |
|---|---|---|---|
| Точка безубыточности | **0.346%** (0.54 млн ₽) | **0.205%** (0.32 млн ₽) | **0.936%** (1.46 млн ₽) |
| MDE при n = 50 000 | 7.72% (12.0 млн ₽) | 7.72% (12.0 млн ₽) | 7.72% (12.0 млн ₽) |
| P(внедрить) при μ | 0.038 | 0.298 | 0.009 |
| RR | 2.56 | 11.17 | 0.18 |
| Тест: чистая прибыль | +0.41 млн ₽ | +1.88 млн ₽ | −0.41 млн ₽ |
| Без теста (вслепую) | 0 | +5.15 млн ₽ | −1.56 млн ₽ |
| Вердикт | **тестировать** | **внедрять без теста** | **не внедрять, доработать** |

**Разбор «почему у H1 прибыль положительна при вероятности внедрения 4%».** Тест — это опция: мы платим фиксированные 0.18 млн ₽ за право внедрить только хорошие варианты. Вклад в ожидание:

- 0.04 млн ₽ — выгода во время теста (половина трафика уже получает улучшение);
- 0.64 млн ₽ — выгода после внедрения, взвешенная вероятностью внедрения: она приходит из редких сценариев, где эффект большой и поэтому внедряется;
- −0.27 млн ₽ — все затраты, включая риск-скорректированные (откат в плохих сценариях).

Итого +0.41 млн ₽. Обсуждение для группы: если у команды есть 10 таких гипотез и трафик на одну, ICE-логика говорит брать в первую очередь H2 (там решение и так очевидно), а H1 — только при свободном трафике.

---

## Упражнение 2. Единица и критерий (20 мин)

### Задача

Для трёх кейсов выбрать единицу рандомизации и единицу анализа:

1. посчитать дизайн-эффект $DE = 1+(m-1)\rho$ и эффективный размер выборки;
2. сравнить наивный MDE (по событиям) с кластерным;
3. выбрать критерий из списка: `z-тест долей`, `Welch по пользователям`, `t-тест по кластерам`;
4. заполнить таблицу «кейс → единица → DE → критерий».

### Входные данные

| Кейс | Что меняем | $m$ (наблюдений в кластере) | $\rho$ (ICC) | $k$ (кластеров на группу) | Наивная ошибка |
|---|---|---|---|---|---|
| Маркетплейс | карточка товара у продавцов | 150 покупателей | 0.15 | 400 продавцов | z-тест по покупателям |
| Соцсеть | рекомендации в ленте | 8 сессий | 0.20 | 100 000 пользователей | тест по сессиям |
| Оффлайн-сеть | выкладка в залах | 1 200 чеков | 0.02 | 40 магазинов | t-тест по чекам |

### Решение

**Ячейка 1. Дизайн-эффект и MDE**

In [ ]:
from scipy.stats import norm

Z_SUM = norm.ppf(0.975) + norm.ppf(0.8)


def design_effect(m, rho):
    """Во сколько раз кластеризация увеличивает дисперсию среднего."""
    return 1 + (m - 1) * rho


def mde_cluster(k, m, rho, sigma=1.0):
    """MDE в единицах стандартного отклонения, с поправкой на кластеры."""
    return Z_SUM * np.sqrt(2 * sigma ** 2 * design_effect(m, rho) / (k * m))


def mde_naive(k, m, sigma=1.0):
    """Тот же MDE, если ошибочно считать наблюдения независимыми."""
    return Z_SUM * np.sqrt(2 * sigma ** 2 / (k * m))


def effective_n(k, m, rho):
    """Сколько независимых наблюдений реально остаётся на группу."""
    return k * m / design_effect(m, rho)


**Ячейка 2. Три кейса и выбор критерия**

In [ ]:
CASES = {
    "Маркетплейс": dict(m=150, rho=0.15, k=400),
    "Соцсеть": dict(m=8, rho=0.20, k=100_000),
    "Оффлайн-сеть": dict(m=1200, rho=0.02, k=40),
}

CHOICE = {          # заполнить самим: единица -> критерий
    "Маркетплейс": "t-тест по средним продавцов",
    "Соцсеть": "Welch по пользователям",
    "Оффлайн-сеть": "t-тест по магазинам",
}

rows = []
for name, c in CASES.items():
    de = design_effect(c["m"], c["rho"])
    rows.append({
        "DE": round(de, 2),
        "номинальный n": c["k"] * c["m"],
        "эффективный n": round(effective_n(c["k"], c["m"], c["rho"])),
        "MDE кластерный": round(mde_cluster(c["k"], c["m"], c["rho"]), 4),
        "MDE наивный": round(mde_naive(c["k"], c["m"]), 4),
        "во сколько раз занижаем": round(mde_cluster(c["k"], c["m"], c["rho"]) / mde_naive(c["k"], c["m"]), 2),
    })

units = pd.DataFrame(rows, index=list(CASES))
units


**Ячейка 3. Критерий приёмки**

In [ ]:
assert abs(design_effect(150, 0.15) - 23.35) < 0.01
assert abs(effective_n(400, 150, 0.15) - 2570) < 5
assert abs(design_effect(8, 0.20) - 2.4) < 1e-9
assert abs(8 / design_effect(8, 0.20) - 3.33) < 0.01          # сессий на пользователя «по факту»
assert abs(design_effect(1200, 0.02) - 24.98) < 1e-9
assert abs(effective_n(40, 1200, 0.02) - 1922) < 5
ratio = mde_cluster(40, 1200, 0.02) / mde_naive(40, 1200)
assert 4.9 < ratio < 5.1                                       # наивный тест занижает ошибку в 5 раз
assert CHOICE == {"Маркетплейс": "t-тест по средним продавцов",
                  "Соцсеть": "Welch по пользователям",
                  "Оффлайн-сеть": "t-тест по магазинам"}

print("Упражнение 2: все проверки пройдены")

### Ответы

| Кейс | Единица рандомизации → анализа | DE | Эффективный n | MDE кластерный / наивный |
|---|---|---|---|---|
| Маркетплейс | продавец → продавец | 23.35 | 2 570 покупателей-эквивалентов | 0.078 / 0.016 |
| Соцсеть | пользователь → пользователь | 2.40 | 333 333 сессий (3.33 на пользователя) | 0.0069 / 0.0044 |
| Оффлайн-сеть | магазин → магазин | 24.98 | 1 922 чека-эквивалента | 0.090 / 0.018 |

Выводы, которые должны прозвучать:

- Кластеризация не «портит» тест, а меняет его: при ICC 0.02 и 1200 чеках в магазине от 48 000 наблюдений остаётся менее 2 000.
- Для соцсети метрика должна быть на пользователя (сессии внутри пользователя зависимы): 8 сессий дают информацию примерно как 3.3 независимых наблюдения.
- Маркетплейс: рандомизировали продавца — отчитываемся эффектом на продавца; эффект на покупателя потребует взвешивания по размеру (обсудить, нужен ли он бизнесу).

---

## Упражнение 3. Подбор α и n (20 мин)

### Задача

Для гипотезы H1 (купон отдельной строкой) построить сетку $\alpha \times n$ и ответить на два вопроса:

1. Где максимум **чистой прибыли** и где максимум **отношения RR** — совпадают ли они и лежит ли оптимум внутри сетки?
2. Какова **цена малой мощности**: Type S и Type M при истинном эффекте +0.05 п.п. и +0.20 п.п.

### Входные данные

$\alpha \in \{0.05, 0.1, 0.2, 0.3, 0.5\}$, $n \in \{10, 25, 50, 100, 150, 200, 300, 400, 500, 600\}$ тыс. на группу. Гипотеза H1 та же: точка безубыточности 0.346%, априорное распределение $0\% \pm 3\%$.

### Решение

**Ячейка 1. Сетка: чистая прибыль и RR**

In [ ]:
NS = [10_000, 25_000, 50_000, 100_000, 150_000, 200_000, 300_000, 400_000, 500_000, 600_000]
ALPHAS = [0.05, 0.1, 0.2, 0.3, 0.5]

net = pd.DataFrame(index=NS, columns=ALPHAS, dtype=float)
rr = pd.DataFrame(index=NS, columns=ALPHAS, dtype=float)

for n in NS:
    for a in ALPHAS:
        r = evaluate(H1, n, a)
        net.loc[n, a] = r["net"] / 1e6
        rr.loc[n, a] = r["rr"]

print("Чистая прибыль, млн ₽")
print(net.round(2).to_string())
print()
print("RR")
print(rr.round(2).to_string())


**Ячейка 2. Тепловая карта (по желанию)**

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, data, title in ((axes[0], net, "Чистая прибыль, млн ₽"), (axes[1], rr, "RR")):
    im = ax.imshow(data.values, aspect="auto", origin="lower", cmap="RdYlGn", vmin=0)
    ax.set_xticks(range(len(ALPHAS)), labels=[str(a) for a in ALPHAS])
    ax.set_yticks(range(len(NS)), labels=[f"{n // 1000}к" for n in NS])
    ax.set_xlabel("α")
    ax.set_ylabel("n на группу")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


**Ячейка 3. Оптимумы**

In [ ]:
best_net_n, best_net_alpha = net.stack().idxmax()
best_rr_n, best_rr_alpha = rr.stack().idxmax()

print(f"Максимум прибыли: alpha = {best_net_alpha}, n = {best_net_n:,}, {net.stack().max():.2f} млн ₽, "
      f"тест {2 * best_net_n / USERS_WEEK:.1f} нед.")
print(f"Максимум RR:      alpha = {best_rr_alpha}, n = {best_rr_n:,}, RR = {rr.stack().max():.2f}, "
      f"прибыль {net.loc[best_rr_n, best_rr_alpha]:.2f} млн ₽")
print(f"Стандартный дизайн: alpha = 0.05, n = 50 000, прибыль {net.loc[50_000, 0.05]:.2f} млн ₽")
print(f"«На всякий случай больше»: alpha = 0.05, n = 600 000, прибыль {net.loc[600_000, 0.05]:.2f} млн ₽")


**Ячейка 4. Цена малой мощности: Type S и Type M**

In [ ]:
def retrodesign(theta, se, alpha=0.05):
    """Power, Type S и Type M для нормальной несмещённой оценки (формулы I.07)."""
    lam = abs(theta) / se
    z = norm.ppf(1 - alpha / 2)
    p_plus = 1 - norm.cdf(z - lam)
    p_minus = norm.cdf(-z - lam)
    power = p_plus + p_minus
    type_s = p_minus / power
    type_m = (lam * (p_plus - p_minus) + norm.pdf(z - lam) + norm.pdf(z + lam)) / (lam * power)
    return power, type_s, type_m


rows = []
for theta in (0.0005, 0.001, 0.002, 0.0039):
    for n in (50_000, 400_000):
        se = np.sqrt(2 * P0 * (1 - P0) / n)
        power, type_s, type_m = retrodesign(theta, se)
        rows.append({"истинный эффект, п.п.": round(100 * theta, 2),
                     "n на группу": n,
                     "мощность": round(power, 2),
                     "Type S": round(type_s, 3),
                     "Type M": round(type_m, 2)})

sm = pd.DataFrame(rows).sort_values(["истинный эффект, п.п.", "n на группу"])
sm


**Ячейка 5. Критерий приёмки**

In [ ]:
assert best_net_alpha == 0.5
assert best_net_n in (100_000, 150_000, 200_000)
assert 1.0 < net.stack().max() < 1.2
assert best_rr_alpha == 0.2
assert 3.3 < rr.stack().max() < 3.6
assert abs(net.loc[50_000, 0.05] - 0.41) < 0.02           # стандартный дизайн
assert net.stack().idxmax()[0] < 600_000                  # оптимум внутри сетки, а не на краю
assert net.loc[600_000, 0.05] < net.stack().max()         # «ещё больше выборки» не спасает
assert net.loc[150_000, 0.5] > 2 * net.loc[50_000, 0.05]  # цена строгого порога

s_small = sm[(sm["истинный эффект, п.п."] == 0.05) & (sm["n на группу"] == 50_000)].iloc[0]
assert s_small["Type M"] > 6 and s_small["Type S"] > 0.1

print("Упражнение 3: все проверки пройдены")

### Ответы

| Что искали | α | n | Результат |
|---|---|---|---|
| Максимум чистой прибыли | 0.50 | 150 000 | +1.09 млн ₽ (6 недель) |
| Максимум RR | 0.20 | 100 000 | RR = 3.45, +0.94 млн ₽ |
| Стандартный дизайн | 0.05 | 50 000 | +0.41 млн ₽ |
| Строгий порог и огромная выборка | 0.05 | 600 000 | +0.55 млн ₽ — больше стандарта, но вдвое хуже оптимума |

Цена малой мощности (конверсия 5%):

| Истинный эффект | n = 50 000: мощность / Type S / Type M | n = 400 000: мощность / Type M |
|---|---|---|
| +0.05 п.п. | 0.07 / 0.155 / **6.54** | 0.18 / 2.43 |
| +0.10 п.п. | 0.11 / 0.032 / 3.36 | 0.54 / 1.36 |
| +0.20 п.п. | 0.31 / 0.001 / 1.79 | 0.98 / 1.01 |
| +0.39 п.п. (= MDE) | 0.81 / ~0 / 1.12 | 1.00 / 1.00 |

Что проговорить: на слабом дизайне значимый результат завышает эффект в 6.5 раза и в 15% случаев переворачивает знак. Увеличение выборки в 8 раз чинит и то, и другое; замена α на 0.05 при малой выборке — нет.

---

## Приложение A. Один ноутбук на все три упражнения

Если удобнее держать один файл: ячейки 1–2 упражнения 1 (движок) переиспользуются в упражнении 3 (`evaluate`, `mde_relative`, `H1`, `P0`, `USERS_WEEK`), а упражнение 2 от движка не зависит — там своя маленькая функция `design_effect`.

Порядок запуска: упражнение 1 → упражнение 2 → упражнение 3.

---

## Приложение B. Типичные ошибки на этой практике

| Ошибка | Что происходит | Как ловить |
|---|---|---|
| Сравнивать тест только с «ничего не делать» | H2 выглядит «тестом», хотя внедрять надо сразу | всегда считать `blind = REV_WEEK × μ × HORIZON` |
| Брать MDE как порог решения | MDE (12 млн ₽) в 22 раза больше точки безубыточности (0.54 млн ₽) | порог = break-even; MDE — только про разрешающую способность |
| Считать мощность по наблюдённому эффекту | «мощность 95%, всё отлично» после значимого результата | Design Analysis: θ извне, не из теста |
| Максимизировать RR, когда важна прибыль | выбирается дешёвый слабый тест | явно назвать критерий до расчёта |
| Анализ по событиям при рандомизации по кластерам | ошибка I рода 26–80% вместо 5% | `DE = 1+(m−1)ρ`, агрегация до кластера |